# m48 lab · Imaging across the EP workflow: blur, contrast, noise and the accuracy of a test

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You describe every modality in one vocabulary. You turn a published resolution into what it does to a thin wall and
to two structures close together, compute where contrast comes from in X-ray, CT and MRI, measure contrast-to-noise
ratio and what averaging and smoothing do to it, image one cross-section of the posterior left atrial wall at four
resolutions, and turn the published accuracy of CT for appendage thrombus into predictive values.

| Part | What you do | Data |
| --- | --- | --- |
| A | Blur and resolution (eq-m48-psf) | published resolutions |
| B | Thin walls at published resolutions (eq-m51-wall) | published resolutions and wall thickness |
| C | Where contrast comes from | our own code (m49, m51, m52) |
| D | Contrast-to-noise ratio and averaging (eq-m48-cnr) | our own simulation |
| E | Smoothing: less noise, less resolution (eq-m48-smooth, eq-m48-visibility) | our own phantom |
| F | One cross-section imaged four ways | our own phantom, published resolutions |
| G | Accuracy of an imaging test (eq-m35-ppv, eq-m48-lr) | published meta-analysis |

Published values: CT section-sensitivity FWHM of 0.34 and 0.64 mm on a photon-counting system (Rajendran et al.
2022); LGE resolution of 1.5–2 mm routinely and about 1 mm for thin walls (Kellman and Arai 2012); clinical PET
resolution of 4–7 mm FWHM (Kinahan and Fletcher 2010); left atrial wall thickness 1.89 mm on average, 0.5–3.5 mm
(Beinart et al. 2011); the accuracy of CT for appendage thrombus against TEE (Romero et al. 2013). Treating each
resolution as the FWHM of a Gaussian is a simplification. Contrasts, noise levels, the esophageal geometry and the
simulated meta-analysis are illustrative. Each part ends with **checks**; no downloads.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import overview as O
from epml import ct as CT, fluoro as F, mri as M, ice as I

rng = np.random.default_rng(48)

## Part A · Blur and resolution

Model each scanner's blur as a Gaussian point-spread function of standard deviation $\sigma$; its full width at half
maximum is $\Gamma = 2\sqrt{2\ln 2}\,\sigma \approx 2.355\,\sigma$ (eq-m48-psf). Two equal points a distance $d$
apart show a dip between them only when $d > 2\sigma$, about $0.85\,\Gamma$. Below, the published resolutions and the
separation each needs.

In [ ]:
print(f"{'resolution':62s} {'FWHM':>6s} {'sigma':>6s} {'2 sigma':>8s}")
for name, fwhm in O.RESOLUTION_MM.items():
    s = float(O.sigma_from_fwhm(fwhm))
    print(f"{name:62s} {fwhm:6.2f} {s:6.3f} {2 * s:8.2f}")

x = np.linspace(-6, 6, 2001)
for d in (1.5, 2.0, 2.355, 3.0):
    plt.plot(x, O.two_point_profile(x, d, 1.0), label=f"d = {d} sigma, dip {O.dip_depth(d, 1.0):.0%}")
plt.xlabel("position (units of sigma)"); plt.ylabel("blurred signal"); plt.legend(fontsize=8); plt.show()

ds = np.linspace(1.0, 4.0, 61)
dips = np.array([O.dip_depth(d, 1.0) for d in ds])
measured = O.measured_fwhm(x, O.gaussian_psf(x, 1.0))
print(f"measured FWHM of a unit-sigma PSF: {measured:.4f} (formula {float(O.fwhm_from_sigma(1.0)):.4f})")
print(f"dip at one FWHM: {O.dip_depth(float(O.fwhm_from_sigma(1.0)), 1.0):.1%}")

In [ ]:
# Checks
assert abs(measured - 2 * np.sqrt(2 * np.log(2))) < 1e-3
assert np.all(dips[ds <= 2.0] == 0) and np.all(dips[ds > 2.05] > 0)
assert 0.06 < O.dip_depth(float(O.fwhm_from_sigma(1.0)), 1.0) < 0.08

Two structures one FWHM apart are just distinguishable: the dip is about 7%. With MRI at 1.5 mm, two thin layers
closer than about 1.3 mm merge into one; at the 0.64-mm photon-counting CT section, about 0.5 mm suffices.

## Part B · Thin walls at published resolutions

A wall of thickness $w$ and unit contrast, blurred, peaks at $\operatorname{erf}(w/(2\sqrt2\,\sigma))$
(eq-m51-wall, derived in m51). Read it two ways: with a fixed threshold at half the contrast, or as
the width at half of the blurred peak. Do this for the thinnest and the mean left atrial wall in one CT study
(Beinart et al. 2011).

In [ ]:
rows = []
print(f"{'wall':>5s} {'FWHM':>5s} {'peak':>6s} {'half-peak read':>15s} {'threshold read':>15s}")
for w in (O.LA_WALL_MM["min"], O.LA_WALL_MM["mean"]):
    for fwhm in O.RESOLUTION_MM.values():
        s = float(O.sigma_from_fwhm(fwhm))
        r = (w, fwhm, O.slab_peak(w, s), O.fwhm_thickness(w, s), O.threshold_thickness(w, s))
        rows.append(r)
        print(f"{r[0]:5.2f} {r[1]:5.2f} {r[2]:6.0%} {r[3]:15.2f} {r[4]:15.2f}")
table = {(r[0], r[1]): r[2:] for r in rows}

In [ ]:
# Checks
assert table[(0.5, 0.34)][0] > 0.9          # the finest CT section keeps most of a 0.5-mm wall
assert table[(0.5, 1.0)][2] == 0.0          # at 1 mm a half-contrast threshold misses it
assert table[(1.89, 1.5)][2] > 1.8          # the mean wall reads correctly at routine MRI resolution
assert table[(1.89, 4.0)][2] == 0.0         # and is missed by a half-contrast threshold at PET resolution
assert all(r[3] >= r[1] - 1e-6 for r in rows)  # the half-peak reading never drops below the blur

A thin wall does not disappear gradually: its peak falls, and a fixed threshold suddenly misses it, while a reading at
half its own peak never reports less than the blur. Neither number is the wall.

## Part C · Where contrast comes from

Blur is half the story; the other half is how different two tissues look. With the 60-keV attenuation data used in
m49 and m51, compute (1) the X-ray contrast of 2 mm and 20 mm of wall against blood, and the photons per pixel
needed for a CNR of 3 (an illustrative target) from eq-m49-cnr; (2) the same with 10 mg/mL of iodine in the blood
(illustrative); (3) Hounsfield units of wall, blood and fat; (4) the LGE signal of infarct when normal myocardium
is nulled, from the T1 values of Bauner et al. (2012) and the m52 code.

In [ ]:
mu_wall, mu_blood = CT.mu_per_mm("soft tissue"), CT.mu_per_mm("blood")
mu_iodine_blood = CT.blood_with_iodine_mu(10.0)
for L in (2.0, 20.0):
    c = float(F.subject_contrast(mu_wall, mu_blood, L))
    print(f"X-ray, {L:4.0f} mm of wall against blood: contrast {abs(c):.3%}, photons for CNR 3: {float(F.photons_for_cnr(abs(c), 3)):.2e}")
c_iod = float(F.subject_contrast(mu_wall, mu_iodine_blood, 2.0))
print(f"X-ray, 2 mm of wall against blood with 10 mg/mL iodine: contrast {abs(c_iod):.2%}, photons for CNR 3: {float(F.photons_for_cnr(abs(c_iod), 3)):.2e}")
hu = {k: float(CT.hounsfield(CT.mu_per_mm(k))) for k in ("soft tissue", "blood", "adipose")}
hu["blood + 10 mg/mL iodine"] = float(CT.hounsfield(mu_iodine_blood))
for k, v in hu.items():
    print(f"CT {k:26s} {v:7.0f} HU")
b = M.BAUNER
lge = M.scar_signal_at_null(b["myocardium"]["t1_post"], b["infarct"]["t1_post"])
print(f"LGE: infarct signal with normal myocardium nulled: {lge:.2f} of full magnetization")

In [ ]:
# Checks
assert abs(float(F.subject_contrast(mu_wall, mu_blood, 2.0))) < 1e-3     # blood and wall: almost no X-ray contrast
assert abs(c_iod) > 50 * abs(float(F.subject_contrast(mu_wall, mu_blood, 2.0)))
assert abs(hu["soft tissue"] - hu["blood"]) < 10 and hu["blood + 10 mg/mL iodine"] - hu["soft tissue"] > 300
assert 0.3 < lge < 0.4

Without contrast agent, X-rays barely tell wall from blood: about 5 HU apart in CT, and a fraction of a percent in a
projection. Iodine raises blood by about 370 HU at 10 mg/mL. MRI builds contrast from relaxation times instead, and
PET from where a tracer goes. Contrast is a choice of physics, agent and timing, not a property of the anatomy.

## Part D · Contrast-to-noise ratio and averaging

$\mathrm{CNR} = |S_A - S_B|/\sigma_n$, and averaging $n$ independent samples divides the noise by $\sqrt n$
(eq-m48-cnr). Measure it on simulated pixels with contrast 1 and noise 1 (illustrative). For photon-limited images,
$\sigma_n$ falls as one over the square root of the counts, so halving the noise needs four times the dose
(eq-m49-cnr).

In [ ]:
ns = (1, 4, 16, 64)
meas = []
for n in ns:
    a = rng.normal(1.0, 1.0, (20000, n)).mean(axis=1)
    bkg = rng.normal(0.0, 1.0, (20000, n)).mean(axis=1)
    meas.append(O.measured_cnr(a, bkg))
    print(f"n = {n:3d}: measured CNR {meas[-1]:.2f}, eq-m48-cnr {float(O.cnr_averaged(1.0, n)):.2f}")
print("photons needed to halve the noise:", 4, "times as many (eq-m49-cnr)")

In [ ]:
# Checks
for n, m in zip(ns, meas):
    assert abs(m / float(O.cnr_averaged(1.0, n)) - 1) < 0.05

## Part E · Smoothing: less noise, less resolution

Smoothing white noise on a grid of spacing $\Delta$ with a Gaussian of standard deviation $\sigma_s$ leaves
$\sigma_n\,\Delta/(2\sqrt\pi\,\sigma_s)$ (eq-m48-smooth) and widens the blur to $\sqrt{\sigma^2 + \sigma_s^2}$. A wall's
CNR becomes $(C/\sigma_n)\operatorname{erf}(w/(2\sqrt2\,\sigma_{\text{tot}}))$ (eq-m48-visibility). Take the mean
wall, 1 mm of blur, 0.1-mm pixels with noise 2 per pixel (illustrative), and an illustrative esophageal wall 2 mm
behind it. Measure the noise on a large patch of noise alone and the contrast on the noise-free phantom, as imaging
physicists do with uniform and contrast phantoms.

In [ ]:
pixel, noise, w = 0.1, 2.0, O.LA_WALL_MM["mean"]
sigma = float(O.sigma_from_fwhm(1.0))
labels, y = O.wall_phantom(pixel=pixel, width=w, gap=2.0, size=(60.0, 50.0))
w_grid = (labels[:, 0] == 2).sum() * pixel                # the wall as drawn on this grid
wall_only = O.image(labels, {2: 1.0}, 1.0, 0.0, pixel, rng)   # noise-free, to measure contrast
raw = wall_only + rng.normal(0.0, noise, labels.shape)      # with noise, to look at
pure = rng.normal(0.0, noise, (1200, 1200))                 # noise alone, to measure noise
print(f"wall drawn {w_grid:.2f} mm thick on the {pixel} mm grid")
res = []
for s_s in (0.25, 0.5, 1.0, 1.5, 2.0):
    tot = float(np.hypot(sigma, s_s))
    noise_meas = ndi.gaussian_filter(pure, s_s / pixel)[100:-100, 100:-100].std()
    sm = ndi.gaussian_filter(wall_only, s_s / pixel, mode="nearest")[:, 0]
    contrast = sm[np.argmin(np.abs(y))] - sm[np.argmin(np.abs(y - 15))]
    both = O.depth_profile(O.image(labels, {2: 1.0, 3: 1.0}, float(O.fwhm_from_sigma(tot)), 0.0, pixel, rng))
    spill = both[np.argmin(np.abs(y))] - O.slab_peak(w_grid, tot)   # what the esophagus adds at the wall's center
    res.append((s_s, noise_meas, float(O.smoothed_noise(noise, pixel, s_s)), contrast / noise_meas,
                float(O.smoothed_wall_cnr(1.0, noise, w_grid, sigma, pixel, s_s)), O.separated(both, y, w, 2.0), spill))
    r = res[-1]
    print(f"smoothing {r[0]:4.2f} mm: noise {r[1]:.3f} (eq {r[2]:.3f}), wall CNR {r[3]:5.1f} (eq {r[4]:5.1f}), "
          f"wall and esophagus separate: {r[5]}, esophagus spill-in {r[6]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].imshow(raw, cmap="gray", extent=[0, 60, y[0], y[-1]], origin="lower", vmin=-3, vmax=4)
ax[0].set_ylim(-8, 6)
ax[0].set_title("raw (1 mm blur, noisy)"); ax[0].set_ylabel("depth (mm)")
ax[1].imshow(ndi.gaussian_filter(raw, 1.0 / pixel), cmap="gray", extent=[0, 60, y[0], y[-1]], origin="lower")
ax[1].set_title("smoothed, 1 mm"); ax[1].set_ylim(-8, 6)
plt.show()

In [ ]:
# Checks
for s_s, nm, ne, cm, ce, sep, spill in res:
    assert abs(nm / ne - 1) < 0.05
    assert abs(cm / ce - 1) < 0.1
assert res[-1][3] > 2 * res[0][3]           # smoothing raises the wall's CNR
assert res[0][5] and not res[-1][5]         # but merges the wall with the esophagus
assert res[0][6] < 1e-3 < 0.05 < res[-1][6]  # and lets the neighbor spill into the wall

Smoothing buys CNR and pays in resolution: the wall becomes easier to see and harder to separate from its
neighbor, and the neighbor's signal spills into it (the partial-volume effect of m54). Averaging along the wall, as the depth profiles in part F do, buys CNR without blurring across it; that
only works when the structure's direction is known.

## Part F · One cross-section imaged four ways

A depth profile through the posterior left atrial wall: blood above, the atrial wall, a fat gap and the esophageal
wall below. Signals are illustrative and chosen so that both walls are brighter than their neighbors; only the blur
changes, set to a photon-counting CT section, the axial resolution of ICE at 7.5 MHz with an illustrative 2-cycle
pulse (eq-m50-axial), routine LGE and best-case PET. Ask two questions: does the wall rise above the blood, and is it
separate from the esophagus?

In [ ]:
ways = {
    "CT, photon-counting section (0.64 mm)": 0.64,
    "ICE, axial, 7.5 MHz, 2 cycles": float(I.axial_resolution_mm(7.5, 2)),
    "MRI LGE, routine (1.5 mm)": 1.5,
    "PET, best case (4 mm)": 4.0,
}
vals = {0: 0.0, 1: 0.4, 2: 1.0, 3: 1.0}
out = {}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for k, wall in enumerate((O.LA_WALL_MM["mean"], O.LA_WALL_MM["min"])):
    labels, y = O.wall_phantom(width=wall, gap=2.0)
    ax[k].plot(y, O.depth_profile(labels == 2) + 0.4 * O.depth_profile(labels == 1) + O.depth_profile(labels == 3),
               color="gray", ls="--", label="true")
    for name, fwhm in ways.items():
        prof = O.depth_profile(O.image(labels, vals, fwhm, 0.3, 0.05, rng))
        clean = O.depth_profile(O.image(labels, vals, fwhm, 0.0, 0.05, rng))
        out[(wall, name)] = (O.wall_above_blood(clean, y, wall, 0.4), O.separated(clean, y, wall, 2.0))
        ax[k].plot(y, prof, label=name)
    ax[k].set_xlim(-9, 5); ax[k].set_xlabel("depth (mm), blood to the right"); ax[k].set_title(f"wall {wall} mm")
ax[0].set_ylabel("signal"); ax[1].legend(fontsize=7); plt.show()
for (wall, name), (above, sep) in out.items():
    print(f"wall {wall:4.2f} mm, {name:40s}: rises {above:5.2f} above blood; separate from esophagus: {sep}")

In [ ]:
# Checks
m, t = O.LA_WALL_MM["mean"], O.LA_WALL_MM["min"]
assert all(out[(m, n)][1] for n in ways if "PET" not in n) and not out[(m, "PET, best case (4 mm)")][1]
assert out[(t, "CT, photon-counting section (0.64 mm)")][0] > 0.2
assert out[(t, "MRI LGE, routine (1.5 mm)")][0] < 0.1

The same anatomy gives four different pictures. The thin wall survives at CT and ICE axial resolution, all but
vanishes into the blood at routine MRI resolution, and at PET resolution the wall and the esophagus merge. This is why
the question decides the modality, and why any number read from an image belongs with the resolution that produced
it.

## Part G · Accuracy of an imaging test

In a meta-analysis of 19 studies (2,955 patients), CT against TEE for left atrial appendage thrombus had pooled
sensitivity 96% and specificity 92%, PPV 41% and NPV 99%, with thrombus in 8.9%; with delayed imaging, 100%, 99%,
92% and 100% (Romero et al. 2013). Turn sensitivity and specificity into predictive values with eq-m35-ppv, and into
likelihood ratios: $\mathrm{LR}^+ = \mathrm{Se}/(1-\mathrm{Sp})$, $\mathrm{LR}^- = (1-\mathrm{Se})/\mathrm{Sp}$, with
post-test odds equal to pre-test odds times LR (eq-m48-lr).

In [ ]:
r = O.ROMERO2013
for key in ("standard", "delayed imaging"):
    a = r[key]
    ppv, npv = O.ppv_npv(a["se"], a["sp"], r["prevalence"])
    lp, ln = O.likelihood_ratios(a["se"], a["sp"])
    print(f"{key:16s}: formula PPV {ppv:.0%} (reported {a['ppv']:.0%}), NPV {npv:.1%} (reported {a['npv']:.0%}), "
          f"LR+ {lp:.0f}, LR- {ln:.3f}")
ppv_std = O.ppv_npv(0.96, 0.92, r["prevalence"])[0]
ppv_del = O.ppv_npv(1.0, 0.99, r["prevalence"])[0]

pis = np.linspace(0.005, 0.5, 200)
plt.plot(100 * pis, 100 * O.ppv_npv(0.96, 0.92, pis)[0], label="PPV, Se 96%, Sp 92%")
plt.plot(100 * pis, 100 * O.ppv_npv(1.0, 0.99, pis)[0], label="PPV, Se 100%, Sp 99%")
plt.scatter([8.9, 8.9], [41, 92], color="k", zorder=3, label="pooled PPV reported")
plt.xlabel("prevalence (%)"); plt.ylabel("PPV (%)"); plt.legend(fontsize=8); plt.show()

# Illustrative: 19 studies with the same Se and Sp but prevalences from 2% to 25%, 150 patients each.
prev = rng.uniform(0.02, 0.25, 19)
ppvs, mean_ppv, at_mean = O.simulate_pooled_ppv(0.96, 0.92, prev, 150, rng)
print(f"mean of per-study PPVs {mean_ppv:.0%}; PPV at the mean prevalence ({prev.mean():.1%}) {at_mean:.0%}")

In [ ]:
# Checks
assert abs(ppv_std - 0.54) < 0.01 and abs(ppv_del - 0.907) < 0.01
assert r["standard"]["ppv"] < ppv_std        # pooled study by study, the reported PPV is lower
assert mean_ppv < at_mean                    # averaging PPVs across prevalences pulls the mean down

The formula gives 54% where the meta-analysis reported a pooled 41%. The two need not agree: each measure was
pooled across studies separately, and PPV depends on each study's prevalence. Because PPV is concave in prevalence,
the average of per-study PPVs lies below the PPV at the average prevalence, as the illustrative simulation shows;
that is one reason, not necessarily the whole one. Either way, a positive CT here is a reason to look again, and a
negative one, at an NPV near 99%, is strong evidence against thrombus at this prevalence. Delayed imaging moves the
PPV to about 91%, close to the reported 92%.

## Where to go next

- In part A, how close can two structures be at the coarser end of routine LGE (2 mm FWHM) and still show a dip?
- In part E, smooth only along the wall (x) and not across it. What happens to the CNR and to the separation?
- In part F, set the blood signal equal to the wall's. Which modality still shows the wall?
- In part G, at what prevalence would the standard CT protocol reach a PPV of 80%?